## 1. Setup Environment

In [ ]:
# Check GPU availability
!nvidia-smi

In [ ]:
# Mount Google Drive
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
# Clone or update repo
import os

REPO_PATH = '/content/Ego4d-LiteSTA'

if os.path.exists(REPO_PATH):
    print('Repo exists, pulling latest...')
    !cd {REPO_PATH} && git pull
else:
    print('Cloning repo...')
    !git clone https://github.com/saeedzns/Ego4d-LiteSTA.git {REPO_PATH}

In [ ]:
# Install dependencies
!pip install -q torch torchvision tqdm transformers pillow pyyaml

In [ ]:
# Set working directory
import sys
os.chdir(f'{REPO_PATH}/local_extraction')
sys.path.insert(0, f'{REPO_PATH}/local_extraction')
print(f'Working directory: {os.getcwd()}')

## 2. Setup Paths

Configure paths to your data on Google Drive.

In [ ]:
# === CONFIGURE THESE PATHS ===

# Path to your VideoMAE checkpoint on Drive
VIDEOMAE_CHECKPOINT = '/content/drive/MyDrive/Ego4d_STA/videomae/checkpoints/videomae_ego_scratch_last.pt'

# Path to extracted frames on Drive
FRAMES_ROOT = '/content/drive/MyDrive/ego4d_data/v2/extracted_frames'

# Path to Track A Stage B manifests (or copy locally)
MANIFESTS_DRIVE = '/content/drive/MyDrive/Ego4d_STA/runs/Track_A'

# Verify paths exist
print('Checking paths...')
print(f'VideoMAE checkpoint: {os.path.exists(VIDEOMAE_CHECKPOINT)}')
print(f'Frames root: {os.path.exists(FRAMES_ROOT)}')
print(f'Manifests: {os.path.exists(MANIFESTS_DRIVE)}')

In [ ]:
# Create local directories and symlinks
!mkdir -p {REPO_PATH}/local_extraction/videomae_local_part/checkpoints_ego_scratch
!mkdir -p {REPO_PATH}/local_extraction/v2
!mkdir -p {REPO_PATH}/local_extraction/runs/Track_A

# Copy/link VideoMAE checkpoint
LOCAL_CKPT = f'{REPO_PATH}/local_extraction/videomae_local_part/checkpoints_ego_scratch/videomae_ego_scratch_last.pt'
if not os.path.exists(LOCAL_CKPT):
    !cp "{VIDEOMAE_CHECKPOINT}" "{LOCAL_CKPT}"
    print(f'Copied checkpoint to {LOCAL_CKPT}')

# Link frames
LOCAL_FRAMES = f'{REPO_PATH}/local_extraction/v2/extracted_frames'
if not os.path.exists(LOCAL_FRAMES):
    !ln -s "{FRAMES_ROOT}" "{LOCAL_FRAMES}"
    print(f'Linked frames to {LOCAL_FRAMES}')

In [ ]:
# Copy Track A manifests (find latest stageB run)
import glob

stageb_runs = sorted(glob.glob(f'{MANIFESTS_DRIVE}/trackA_stageB_*'))
if stageb_runs:
    LATEST_STAGEB = stageb_runs[-1]
    LOCAL_STAGEB = f'{REPO_PATH}/local_extraction/runs/Track_A/{os.path.basename(LATEST_STAGEB)}'
    
    if not os.path.exists(LOCAL_STAGEB):
        !cp -r "{LATEST_STAGEB}" "{LOCAL_STAGEB}"
        print(f'Copied manifests to {LOCAL_STAGEB}')
    else:
        print(f'Manifests already exist: {LOCAL_STAGEB}')
    
    # Check manifest files
    !ls -la "{LOCAL_STAGEB}"/*.jsonl 2>/dev/null || echo 'No jsonl files found'
else:
    print('ERROR: No Track A Stage B runs found!')
    print(f'Please upload manifests to: {MANIFESTS_DRIVE}/trackA_stageB_YYYYMMDD_HHMMSS/')

## 3. Verify Setup

In [ ]:
# Test config loading
from core import load_config

cfg = load_config('trackB_videomae_ego', colab=True)

print('=== VideoMAE Config ===')
print(f'video_backbone: {cfg.get("model.tokenizer.video_backbone")}')
print(f'epochs: {cfg.get("training.epochs")}')
print(f'batch_size: {cfg.get("training.batch_size")}')
print(f'lr: {cfg.get("training.lr")}')
print(f'projector_in_dim: {cfg.get("model.projector.in_dim")}')

In [ ]:
# Test VideoMAE loading
import torch

ckpt_path = f'{REPO_PATH}/local_extraction/videomae_local_part/checkpoints_ego_scratch/videomae_ego_scratch_last.pt'
ckpt = torch.load(ckpt_path, map_location='cpu')

print(f'Checkpoint keys: {list(ckpt.keys())}')
print(f'Epoch: {ckpt.get("epoch", "N/A")}')
if 'config' in ckpt:
    print(f'Hidden size: {ckpt["config"].get("hidden_size", "N/A")}')
    print(f'Num frames: {ckpt["config"].get("num_frames", "N/A")}')

## 4. Run Demo (Quick Test)

In [ ]:
# Run demo to verify pipeline works
!python -m trackB.trackB_train_loader --config trackB_videomae_ego --demo

## 5. Full Training

In [ ]:
# Full training with VideoMAE
# Adjust epochs/batch_size as needed

!python -m trackB.trackB_train_loader \
    --config trackB_videomae_ego \
    --epochs 20 \
    --batch_size 8

In [ ]:
# Alternative: Fewer epochs for testing
# !python -m trackB.trackB_train_loader --config trackB_videomae_ego --epochs 5 --batch_size 8

## 6. Save Results to Drive

In [ ]:
# Copy results back to Drive
RESULTS_DEST = '/content/drive/MyDrive/Ego4d_STA/runs/Track_B_VideoMAE'
!mkdir -p "{RESULTS_DEST}"

# Copy checkpoints and logs
!cp -r {REPO_PATH}/local_extraction/runs/Track_B/* "{RESULTS_DEST}/"

print(f'Results saved to: {RESULTS_DEST}')
!ls -la "{RESULTS_DEST}"

## 7. Compare with ResNet18 Baseline (Optional)

In [ ]:
# Run ResNet18 baseline for comparison
# !python -m trackB.trackB_train_loader --config trackB --epochs 20 --batch_size 8